# Perceptron & Neural Networks — Introduction with PyTorch

---

## Copyright and License

This Jupyter Notebook is provided for educational and research purposes.

## Disclaimer

This notebook is provided "as is", without warranty of any kind, express or implied.  
All analyses and interpretations are for educational and research purposes only.

## Dataset Note

**Dataset:** Moons (synthetic binary classification)  
**Source:** `sklearn.datasets.make_moons`  
**License:** BSD 3-Clause (scikit-learn)

---

## Abstract

This notebook introduces the fundamental building blocks of neural networks — the **perceptron** and the **neuron** — and implements them using **PyTorch**. Starting from the biological inspiration, we build intuition around weights, biases, and activation functions, then train a single-layer and a multi-layer perceptron (MLP) on a synthetic binary classification task.

The notebook covers PyTorch tensor basics, the `nn.Module` API, the training loop, and model evaluation, providing a solid foundation for deeper architectures.

---

## Executive Summary

A single-layer perceptron and a two-layer MLP are trained on the `make_moons` dataset (1,000 samples, 2 features, binary target). We explore the limitations of a linear perceptron on non-linearly separable data and show how adding a hidden layer with a non-linear activation resolves this. Models are evaluated using accuracy and decision boundary visualisation.

---

## Dataset Relevance

The `make_moons` dataset generates two interleaving half-circles — a classic non-linearly separable problem. It is ideal for demonstrating why a single perceptron (linear classifier) fails and why depth + non-linearity matters in neural networks.

---

## Table of Contents

1. [Import Libraries](#1-import-libraries)  
2. [PyTorch Fundamentals — Tensors](#2-pytorch-fundamentals--tensors)  
3. [The Perceptron — Theory](#3-the-perceptron--theory)  
4. [Load & Explore Dataset](#4-load--explore-dataset)  
5. [Preprocessing & DataLoaders](#5-preprocessing--dataloaders)  
6. [Single-Layer Perceptron](#6-single-layer-perceptron)  
7. [Training the Perceptron](#7-training-the-perceptron)  
8. [Multi-Layer Perceptron (MLP)](#8-multi-layer-perceptron-mlp)  
9. [Training the MLP](#9-training-the-mlp)  
10. [Model Evaluation & Decision Boundaries](#10-model-evaluation--decision-boundaries)  
11. [Conclusions](#11-conclusions)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version : {torch.__version__}')
print(f'Using device    : {device}')

## 2. PyTorch Fundamentals — Tensors

PyTorch is built around **tensors** — multi-dimensional arrays similar to NumPy arrays, but with GPU support and automatic differentiation (`autograd`).  
Understanding tensors is the first step before building any neural network.

| Concept | Description |
|---|---|
| `torch.Tensor` | Core data structure (n-dimensional array) |
| `dtype` | Data type: `float32`, `int64`, `bool`, … |
| `shape` | Dimensions of the tensor |
| `device` | Where the tensor lives: `cpu` or `cuda` |
| `requires_grad` | Whether to track gradients for autograd |

In [ ]:
# --- Creating tensors ---
scalar   = torch.tensor(3.14)
vector   = torch.tensor([1.0, 2.0, 3.0])
matrix   = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
zeros    = torch.zeros(3, 4)
ones     = torch.ones(2, 3)
rand_t   = torch.rand(2, 3)          # uniform [0, 1)
randn_t  = torch.randn(2, 3)         # standard normal
arange_t = torch.arange(0, 10, 2)   # like np.arange

print('scalar  :', scalar,  '| ndim:', scalar.ndim,  '| shape:', scalar.shape)
print('vector  :', vector,  '| ndim:', vector.ndim,  '| shape:', vector.shape)
print('matrix  :\n', matrix, '| ndim:', matrix.ndim, '| shape:', matrix.shape)
print('zeros   :\n', zeros)
print('rand    :\n', rand_t)

In [ ]:
# --- Tensor operations ---
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print('Addition       :', a + b)
print('Element-wise * :', a * b)
print('Dot product    :', torch.dot(a, b))
print('Matrix multiply:', torch.mm(matrix, matrix.T))
print('Mean           :', a.mean(), '| Sum:', a.sum(), '| Max:', a.max())

In [ ]:
# --- Reshaping & indexing ---
t = torch.arange(12, dtype=torch.float32)
print('Original :', t.shape)
t_reshaped = t.reshape(3, 4)
print('Reshaped :', t_reshaped.shape)
print('Row 0    :', t_reshaped[0])
print('Col 1    :', t_reshaped[:, 1])
print('Squeeze  :', t_reshaped.unsqueeze(0).shape, '->', t_reshaped.unsqueeze(0).squeeze(0).shape)

In [ ]:
# --- Autograd: automatic differentiation ---
# PyTorch tracks operations on tensors with requires_grad=True
# and computes gradients via backpropagation.

x = torch.tensor(2.0, requires_grad=True)
y = x ** 3 + 2 * x          # y = x^3 + 2x  =>  dy/dx = 3x^2 + 2
y.backward()                 # compute gradient

print(f'y = x^3 + 2x at x=2  =>  y = {y.item():.1f}')
print(f'dy/dx at x=2          =>  {x.grad.item():.1f}  (expected: {3*4+2})')

In [ ]:
# --- NumPy interoperability ---
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)
back_to_np     = tensor_from_np.numpy()

print('NumPy -> Tensor:', tensor_from_np)
print('Tensor -> NumPy:', back_to_np)

## 3. The Perceptron — Theory

### Biological Inspiration

The **perceptron** (Rosenblatt, 1958) is the simplest model of a biological neuron.  
A neuron receives signals through **dendrites**, processes them in the **cell body**, and fires an output through the **axon** if the total signal exceeds a threshold.

### Mathematical Model

Given inputs $x_1, x_2, \ldots, x_n$, weights $w_1, w_2, \ldots, w_n$, and a bias $b$:

$$z = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w}^\top \mathbf{x} + b$$

$$\hat{y} = f(z)$$

where $f$ is an **activation function** that introduces non-linearity.

### Common Activation Functions

| Name | Formula | Range | Use case |
|---|---|---|---|
| Step | $\mathbf{1}[z \geq 0]$ | {0, 1} | Original perceptron |
| Sigmoid | $\frac{1}{1+e^{-z}}$ | (0, 1) | Binary output |
| ReLU | $\max(0, z)$ | $[0, \infty)$ | Hidden layers |
| Tanh | $\tanh(z)$ | (-1, 1) | Hidden layers |

### Limitation: Linear Separability

A single perceptron can only learn **linearly separable** patterns.  
For non-linear problems (like XOR or two moons), we need **multiple layers** — a Multi-Layer Perceptron (MLP).

In [ ]:
# Visualise common activation functions
z = torch.linspace(-5, 5, 200)

activations = {
    'Sigmoid': torch.sigmoid(z),
    'ReLU':    torch.relu(z),
    'Tanh':    torch.tanh(z),
    'Step':    (z >= 0).float(),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
colors = sns.color_palette('muted', 4)

for ax, (name, vals), color in zip(axes, activations.items(), colors):
    ax.plot(z.numpy(), vals.numpy(), color=color, linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_title(name)
    ax.set_xlabel('z')
    ax.set_ylabel('f(z)')

plt.suptitle('Activation Functions', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Load & Explore Dataset

In [ ]:
# Generate the moons dataset
# Two interleaving half-circles — non-linearly separable
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=SEED)

df = pd.DataFrame(X_np, columns=['feature_1', 'feature_2'])
df['label'] = y_np

print(f'Shape       : {df.shape}')
print(f'Classes     : {df["label"].unique()}')
print(f'Class counts:\n{df["label"].value_counts()}')
df.head()

In [ ]:
# Visualise the dataset
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter plot
for label, color, marker in zip([0, 1], ['steelblue', 'tomato'], ['o', 's']):
    mask = df['label'] == label
    axes[0].scatter(df.loc[mask, 'feature_1'], df.loc[mask, 'feature_2'],
                    c=color, marker=marker, alpha=0.6, edgecolors='k', linewidths=0.3,
                    label=f'Class {label}')
axes[0].set_title('Moons Dataset')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()

# Class distribution
counts = df['label'].value_counts().sort_index()
axes[1].bar([f'Class {i}' for i in counts.index], counts.values,
            color=['steelblue', 'tomato'], edgecolor='k', linewidth=0.5)
axes[1].set_title('Class Distribution')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=10)

plt.suptitle('Moons Dataset — Overview', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for i, feat in enumerate(['feature_1', 'feature_2']):
    for label, color in zip([0, 1], ['steelblue', 'tomato']):
        axes[i].hist(df.loc[df['label'] == label, feat], bins=30,
                     alpha=0.6, color=color, label=f'Class {label}', edgecolor='k', linewidth=0.3)
    axes[i].set_title(f'Distribution of {feat}')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Feature Distributions by Class', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Preprocessing & DataLoaders

In [ ]:
# Train / test split (80/20, stratified)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=SEED, stratify=y_np
)

# Feature scaling — important for gradient-based optimisation
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_test_np  = scaler.transform(X_test_np)

print(f'Train size : {X_train_np.shape[0]}')
print(f'Test size  : {X_test_np.shape[0]}')

In [ ]:
# Convert to PyTorch tensors
X_train = torch.tensor(X_train_np, dtype=torch.float32).to(device)
X_test  = torch.tensor(X_test_np,  dtype=torch.float32).to(device)
y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1).to(device)  # (N, 1)
y_test  = torch.tensor(y_test_np,  dtype=torch.float32).unsqueeze(1).to(device)

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('dtype        :', X_train.dtype)
print('device       :', X_train.device)

In [ ]:
# Wrap in TensorDataset and DataLoader for mini-batch training
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Test batches  : {len(test_loader)}')

# Inspect one batch
X_batch, y_batch = next(iter(train_loader))
print(f'Batch X shape : {X_batch.shape}')
print(f'Batch y shape : {y_batch.shape}')

## 6. Single-Layer Perceptron

A **single-layer perceptron** is a linear model: one `nn.Linear` layer followed by a sigmoid activation.  
It computes: $\hat{y} = \sigma(\mathbf{w}^\top \mathbf{x} + b)$

In PyTorch, every model is a subclass of `nn.Module`. The two key methods are:
- `__init__`: define layers
- `forward`: define the forward pass (how data flows through the network)

PyTorch automatically computes the backward pass (gradients) via `autograd`.

In [ ]:
class SinglePerceptron(nn.Module):
    """Single-layer perceptron: Linear -> Sigmoid."""

    def __init__(self, input_dim: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # one output neuron

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))


perceptron = SinglePerceptron(input_dim=2).to(device)
print(perceptron)
print(f'\nTotal parameters: {sum(p.numel() for p in perceptron.parameters())}')

# Inspect initial weights and bias
for name, param in perceptron.named_parameters():
    print(f'  {name}: shape={param.shape}, values={param.data}')

In [ ]:
# Test a forward pass with a single sample
sample = X_train[0].unsqueeze(0)   # shape (1, 2)
output = perceptron(sample)
print(f'Input  : {sample}')
print(f'Output : {output.item():.4f}  (probability of class 1)')
print(f'Pred   : {int(output.item() >= 0.5)}')

## 7. Training the Perceptron

The **training loop** in PyTorch follows a consistent pattern:

1. **Forward pass** — compute predictions
2. **Compute loss** — measure error (Binary Cross-Entropy for binary classification)
3. **Zero gradients** — clear accumulated gradients from previous step
4. **Backward pass** — compute gradients via `loss.backward()`
5. **Update weights** — optimizer step (`optimizer.step()`)

We use **Binary Cross-Entropy Loss**:
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

In [ ]:
def train_model(model, train_loader, test_loader, epochs=100, lr=0.01):
    """Generic training loop for binary classification."""
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        train_loss, train_correct = 0.0, 0
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            preds = model(X_b)
            loss  = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            train_loss    += loss.item() * X_b.size(0)
            train_correct += ((preds >= 0.5).float() == y_b).sum().item()

        # --- Evaluation ---
        model.eval()
        test_loss, test_correct = 0.0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                preds = model(X_b)
                loss  = criterion(preds, y_b)
                test_loss    += loss.item() * X_b.size(0)
                test_correct += ((preds >= 0.5).float() == y_b).sum().item()

        n_train, n_test = len(train_loader.dataset), len(test_loader.dataset)
        history['train_loss'].append(train_loss / n_train)
        history['test_loss'].append(test_loss  / n_test)
        history['train_acc'].append(train_correct / n_train)
        history['test_acc'].append(test_correct  / n_test)

        if epoch % 20 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {history["train_loss"][-1]:.4f} | '
                  f'Train Acc: {history["train_acc"][-1]:.4f} | '
                  f'Test Acc: {history["test_acc"][-1]:.4f}')

    return history

In [ ]:
# Train the single-layer perceptron
perceptron = SinglePerceptron(input_dim=2).to(device)
torch.manual_seed(SEED)

print('=== Training Single-Layer Perceptron ===')
history_perc = train_model(perceptron, train_loader, test_loader, epochs=100, lr=0.1)

In [ ]:
# Plot training curves
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].plot(epochs, history['test_loss'],  label='Test Loss', linestyle='--')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('BCE Loss')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='Train Acc')
    axes[1].plot(epochs, history['test_acc'],  label='Test Acc', linestyle='--')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_history(history_perc, 'Single-Layer Perceptron — Training Curves')

## 8. Multi-Layer Perceptron (MLP)

A **Multi-Layer Perceptron** adds one or more **hidden layers** between input and output.  
Each hidden layer applies a linear transformation followed by a non-linear activation (ReLU here).

Architecture:
```
Input (2)  ->  Linear(2, 16)  ->  ReLU  ->  Linear(16, 8)  ->  ReLU  ->  Linear(8, 1)  ->  Sigmoid
```

The hidden layers allow the network to learn **non-linear decision boundaries**, solving the limitation of the single perceptron.

In [ ]:
class MLP(nn.Module):
    """Two-hidden-layer MLP for binary classification."""

    def __init__(self, input_dim: int, hidden1: int = 16, hidden2: int = 8):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


mlp = MLP(input_dim=2).to(device)
print(mlp)
print(f'\nTotal parameters: {sum(p.numel() for p in mlp.parameters())}')

## 9. Training the MLP

In [ ]:
# Train the MLP — using Adam optimizer (adaptive learning rate)
mlp = MLP(input_dim=2).to(device)
torch.manual_seed(SEED)

# Override train_model to use Adam
def train_model_adam(model, train_loader, test_loader, epochs=100, lr=0.01):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            preds = model(X_b)
            loss  = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            train_loss    += loss.item() * X_b.size(0)
            train_correct += ((preds >= 0.5).float() == y_b).sum().item()

        model.eval()
        test_loss, test_correct = 0.0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                preds = model(X_b)
                loss  = criterion(preds, y_b)
                test_loss    += loss.item() * X_b.size(0)
                test_correct += ((preds >= 0.5).float() == y_b).sum().item()

        n_train, n_test = len(train_loader.dataset), len(test_loader.dataset)
        history['train_loss'].append(train_loss / n_train)
        history['test_loss'].append(test_loss  / n_test)
        history['train_acc'].append(train_correct / n_train)
        history['test_acc'].append(test_correct  / n_test)

        if epoch % 20 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Train Loss: {history["train_loss"][-1]:.4f} | '
                  f'Train Acc: {history["train_acc"][-1]:.4f} | '
                  f'Test Acc: {history["test_acc"][-1]:.4f}')
    return history

print('=== Training MLP ===')
history_mlp = train_model_adam(mlp, train_loader, test_loader, epochs=100, lr=0.01)

In [ ]:
plot_history(history_mlp, 'MLP — Training Curves')

In [ ]:
# Compare final test accuracy: Perceptron vs MLP
fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Single Perceptron', 'MLP']
values = [history_perc['test_acc'][-1], history_mlp['test_acc'][-1]]
bars = ax.bar(labels, values, color=sns.color_palette('muted', 2), edgecolor='k', linewidth=0.5)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Test Accuracy')
ax.set_title('Perceptron vs MLP — Final Test Accuracy')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## 10. Model Evaluation & Decision Boundaries

In [ ]:
def get_predictions(model, X_tensor):
    model.eval()
    with torch.no_grad():
        probs = model(X_tensor).cpu().numpy().flatten()
    return (probs >= 0.5).astype(int), probs

y_pred_perc, _ = get_predictions(perceptron, X_test)
y_pred_mlp,  _ = get_predictions(mlp,        X_test)
y_true = y_test.cpu().numpy().flatten().astype(int)

print('=== Single Perceptron ===')
print(f'Test Accuracy: {accuracy_score(y_true, y_pred_perc):.4f}')
print(classification_report(y_true, y_pred_perc, target_names=['Class 0', 'Class 1']))

print('=== MLP ===')
print(f'Test Accuracy: {accuracy_score(y_true, y_pred_mlp):.4f}')
print(classification_report(y_true, y_pred_mlp, target_names=['Class 0', 'Class 1']))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, preds, title in zip(axes,
                             [y_pred_perc, y_pred_mlp],
                             ['Single Perceptron', 'MLP']):
    cm = confusion_matrix(y_true, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Class 0', 'Class 1']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title} — Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
def plot_decision_boundary(model, X_np, y_np, title, ax, scaler=None):
    """Plot the decision boundary of a trained model."""
    h = 0.02
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]

    if scaler:
        grid = scaler.transform(grid)

    grid_tensor = torch.tensor(grid, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        Z = model(grid_tensor).cpu().numpy().reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=1.5)
    scatter = ax.scatter(X_np[:, 0], X_np[:, 1], c=y_np,
                         cmap='RdBu', edgecolors='k', linewidths=0.3, s=20, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Use original (unscaled) coordinates for the grid, pass scaler for transform
X_test_orig = scaler.inverse_transform(X_test.cpu().numpy())
X_all_orig  = scaler.inverse_transform(np.vstack([X_train.cpu().numpy(), X_test.cpu().numpy()]))
y_all       = np.concatenate([y_train.cpu().numpy().flatten(), y_test.cpu().numpy().flatten()])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(perceptron, X_all_orig, y_all, 'Single Perceptron — Decision Boundary', axes[0], scaler)
plot_decision_boundary(mlp,        X_all_orig, y_all, 'MLP — Decision Boundary',               axes[1], scaler)

plt.suptitle('Decision Boundaries', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Conclusions

- PyTorch tensors are the core data structure: they support GPU acceleration, automatic differentiation, and seamless NumPy interoperability.
- A **single-layer perceptron** is a linear classifier. It can only learn linearly separable boundaries, which is insufficient for the `make_moons` dataset.
- Adding **hidden layers with non-linear activations** (ReLU) enables the MLP to learn curved, complex decision boundaries — dramatically improving accuracy on non-linear problems.
- The **training loop** in PyTorch is explicit and transparent: forward pass → loss → zero_grad → backward → optimizer step.
- **Adam** optimizer generally converges faster than plain SGD for small networks.
- The decision boundary plots visually confirm the perceptron's linear limitation vs the MLP's non-linear capability.
- This foundation — tensors, `nn.Module`, loss functions, optimizers, and the training loop — scales directly to deeper architectures like CNNs and Transformers.